# Upload IRIS FDM data to filament

Uploads a level of a mosaic to filament so `iris_prep` can be applied there.
Used twice per mosaic: level 1 (before
`apply_iris_prep_through_bg_subtraction.pro`) and level 1.2 (before
`apply_remaining_iris_prep.pro`).

Transfers use **rsync**, so a run interrupted by a VPN drop resumes instead of
starting over. On Windows rsync is reached through WSL; authentication is by
SSH key. **Never put a password in this notebook.**

> Storage: `/disk/data` on filament runs at capacity. Delete the previous level
> from filament once the next one has been produced *and downloaded and verified
> locally*. `D:` is the archive and keeps every level.


In [ ]:
from iris_mosaics import MosaicConfig
from iris_mosaics import transfer


Set the mosaic and which level is being uploaded.


In [ ]:
cfg = MosaicConfig.load('20240811')   # <-- the only per-mosaic edit needed
level = 'level_12'   # 'level_1' before iris_prep part 1; 'level_12' before part 2

local_dir = cfg.level_path(level)
remote_dir = f'{cfg.remote_root}/{level}'

print(f'local  : {local_dir}')
print(f'remote : {remote_dir}')
print(f'size   : {transfer.local_bytes(local_dir) / 1e9:.1f} GB in {len(cfg.files(level))} files')


Check filament has room before starting. This raises rather than
part-filling a shared disk.


In [ ]:
print(f'free on filament: {transfer.remote_free_bytes() / 1e9:.0f} GB')
transfer.check_room(local_dir)


Dry run first — see what would move without moving it.

Pass `size_only=True` when reconciling against a copy that was uploaded before
this notebook existed (by scp or pysftp): those files have mtimes that do not
match the local ones, so the default size+mtime comparison would re-send all of
them even though the data is identical.


In [ ]:
transfer.push(local_dir, remote_dir, dry_run=True,
              extra=('--stats', '--no-progress'));


Create the destination and transfer. Hours; safe to re-run — rsync
picks up where it left off.


In [ ]:
transfer._ssh(f'mkdir -p {remote_dir}')


In [ ]:
%%time
transfer.push(local_dir, remote_dir);


Verify the file count matches before moving on.


In [ ]:
local_n = len(cfg.files(level))
remote_n = transfer.remote_count(remote_dir)
print(f'local {local_n}, remote {remote_n}')
assert remote_n == local_n, 'file count mismatch'
